In [7]:
import pandas as pd

df = pd.read_csv('data/factkb/fact_kb_eval_07_04_26.csv')
df.shape

(623, 16)

In [11]:


def build_majority_df(
    df: pd.DataFrame,
    vote_col: str = "Your_annotation_is_supported",
    label_col: str = "FactKB_label",
) -> pd.DataFrame:
    """Group by doc_id and compute majority vote for support labels."""
    if label_col not in df.columns and "factk_label" in df.columns:
        label_col = "factk_label"

    def majority_vote(series: pd.Series):
        # Pick the most frequent non-null label; ties fall back to idxmax order.
        counts = series.dropna().value_counts()
        if counts.empty:
            return None
        return counts.idxmax()

    grouped = (
        df.groupby("doc_id", as_index=False)
        .agg(
            document=("document", "first"),
            gold_summary=("gold_summary", "first"),
            generated_summary=("generated_summary", "first"),
            factk_label=(label_col, "first"),
            factkb_score=("FactKB_score", "first"),
            new_is_supported=(vote_col, majority_vote),
        )
    )

    return grouped

# Example usage:
majority_df = build_majority_df(df)
majority_df.head()

,doc_id,document,gold_summary,generated_summary,factk_label,factkb_score,new_is_supported
0,12396322,"Alex McKechnie, then 16, was in the crowd for ...",It was 50 years ago today that The Beatles pla...,The Cavern Club in Liverpool was the first gig...,0,0.042768,True
1,18298438,It is believed to be the first alleged breach ...,A suspected terrorist has been charged with br...,A 25-year-old man has appeared in court charge...,0,0.277178,True
2,19341954,Concern is mounting about the potential of a v...,The death of Ethiopian Prime Minister Meles Ze...,The death of Ethiopia's Prime Minister Meles Z...,0,0.173333,True
3,23606507,The Â£21.5m Castle Mill development at Port Me...,A campaign group has been granted a hearing to...,An environmental group has launched a legal ch...,0,0.003978,True
4,27119429,The fine follows the conviction of former RBS ...,Royal Bank of Scotland (RBS) has been fined HK...,Royal Bank of Scotland (RBS) has been fined a ...,0,0.286993,True


In [10]:
majority_df.shape

(100, 6)

In [12]:
# save to a csv
majority_df.to_csv('data/factkb/fact_kb_majority_vote.csv', index=False)

In [16]:
import math

def compare_factkb_to_gt(
    df: pd.DataFrame,
    pred_col: str = "factk_label",
    gt_col: str = "new_is_supported",
):
    """Compute classification metrics using new_is_supported as ground truth."""

    def to_binary(value):
        if value is None or (isinstance(value, float) and math.isnan(value)):
            return None
        if isinstance(value, (int, bool)):
            return int(value)
        if isinstance(value, float):
            return int(value)
        text = str(value).strip().lower()
        if text in {"true", "1", "yes", "supported", "support", "entails"}:
            return 1
        if text in {"false", "0", "no", "not supported", "unsupported", "contradiction", "refutes"}:
            return 0
        return None

    work = df[[pred_col, gt_col]].copy()
    work["pred_bin"] = work[pred_col].map(to_binary)
    work["gt_bin"] = work[gt_col].map(to_binary)
    work = work.dropna(subset=["pred_bin", "gt_bin"])

    tp = int(((work["pred_bin"] == 1) & (work["gt_bin"] == 1)).sum())
    tn = int(((work["pred_bin"] == 0) & (work["gt_bin"] == 0)).sum())
    fp = int(((work["pred_bin"] == 1) & (work["gt_bin"] == 0)).sum())
    fn = int(((work["pred_bin"] == 0) & (work["gt_bin"] == 1)).sum())

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    balanced_accuracy = (recall + tnr) / 2

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": accuracy.__round__(4),
        "precision": precision.__round__(4),
        "recall": recall.__round__(4),
        "f1": f1.__round__(4),
        "specificity": specificity.__round__(4),
        "balanced_accuracy": balanced_accuracy.__round__(4),
    }

# Example usage:
metrics = compare_factkb_to_gt(majority_df)
metrics

{'tp': 17,
 'tn': 28,
 'fp': 1,
 'fn': 54,
 'accuracy': 0.45,
 'precision': 0.9444,
 'recall': 0.2394,
 'f1': 0.382,
 'specificity': 0.9655,
 'balanced_accuracy': 0.6025}